In [18]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.float_format', '{:,.2f}'.format)

BASE_V2010 = Path('../hcris-data/HCRIS_v2010')
HCRIS_IN   = Path('../hwk4/data/output/hcris_clean.csv')
KFF_IN     = Path('data/input/kff_expansion.csv')
OUT_DIR    = Path('data/output')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete.')

Setup complete.


In [19]:
hcris = pd.read_csv(HCRIS_IN, low_memory=False)
hcris['provider_number'] = (
    hcris['provider_number'].astype(str).str.strip().str.zfill(6)
)
hcris['year'] = hcris['year'].astype(int)

print(f'HCRIS clean loaded:  {len(hcris):,} rows')
print(f'Years:               {sorted(hcris["year"].unique())}')
print(f'Unique providers:    {hcris["provider_number"].nunique():,}')

HCRIS clean loaded:  64,259 rows
Years:               [np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)]
Unique providers:    6,860


In [20]:
MEDICARE_STATE_CODES = {
    '01': 'AL', '02': 'AK', '03': 'AZ', '04': 'AR', '05': 'CA',
    '06': 'CO', '07': 'CT', '08': 'DE', '09': 'FL', '10': 'GA',
    '11': 'HI', '12': 'ID', '13': 'IL', '14': 'IN', '15': 'IA',
    '16': 'KS', '17': 'KY', '18': 'LA', '19': 'ME', '20': 'MD',
    '21': 'MA', '22': 'MI', '23': 'MN', '24': 'MS', '25': 'MO',
    '26': 'MT', '27': 'NE', '28': 'NV', '29': 'NH', '30': 'NJ',
    '31': 'NM', '32': 'NY', '33': 'NC', '34': 'ND', '35': 'OH',
    '36': 'OK', '37': 'OR', '38': 'PA', '39': 'RI', '40': 'SC',
    '41': 'SD', '42': 'TN', '43': 'TX', '44': 'UT', '45': 'VT',
    '46': 'VA', '47': 'WA', '48': 'WV', '49': 'WI', '50': 'WY',
}

hcris['state'] = hcris['provider_number'].str[:2].map(MEDICARE_STATE_CODES)

n_before = len(hcris)
hcris = hcris.dropna(subset=['state']).copy()
print(f'Dropped non-50-state rows: {n_before - len(hcris):,}')
print(f'Remaining rows:            {len(hcris):,}')

Dropped non-50-state rows: 3,429
Remaining rows:            60,830


In [21]:
S10_VARS = {
    'tot_uncomp_care_charges':      ('S100000', '02000', '00300'),
    'tot_uncomp_care_partial_pmts': ('S100000', '02200', '00300'),
    'bad_debt':                     ('S100000', '02800', '00100'),
}

def get_v2010_paths(year):
    folder = BASE_V2010 / f'HospitalFY{year}'
    nmrc = next((p for p in [
        folder / f'hosp10_{year}_NMRC.CSV',
        folder / f'HOSP10_{year}_nmrc.csv',
    ] if p.exists()), None)
    rpt = next((p for p in [
        folder / f'hosp10_{year}_RPT.CSV',
        folder / f'HOSP10_{year}_rpt.csv',
    ] if p.exists()), None)
    return nmrc, rpt

RPT_COLS = [
    'rpt_rec_num', 'prvdr_ctrl_type_cd', 'provider_number', 'npi',
    'rpt_stus_cd', 'fy_bgn_dt', 'fy_end_dt', 'proc_dt',
    'initl_rpt_sw', 'last_rpt_sw', 'trnsmtl_num', 'fi_num',
    'adr_vndr_cd', 'fi_creat_dt', 'util_cd', 'npr_dt',
    'spec_ind', 'fi_rcpt_dt'
]
NMRC_COLS = ['rpt_rec_num', 'wksht_cd', 'line_num', 'clmn_num', 'itm_val_num']

print('Variable map and functions defined.')

Variable map and functions defined.


In [22]:
def extract_s10_year(year):
    nmrc_path, rpt_path = get_v2010_paths(year)
    if nmrc_path is None or rpt_path is None:
        print(f'SKIP {year} — files not found')
        return None

    rpt = pd.read_csv(rpt_path, header=None, names=RPT_COLS, dtype=str)
    rpt['rpt_rec_num']     = pd.to_numeric(rpt['rpt_rec_num'],  errors='coerce')
    rpt['fy_bgn_dt']       = pd.to_datetime(rpt['fy_bgn_dt'],   errors='coerce')
    rpt['fy_end_dt']       = pd.to_datetime(rpt['fy_end_dt'],   errors='coerce')
    rpt['fi_creat_dt']     = pd.to_datetime(rpt['fi_creat_dt'], errors='coerce')
    rpt['provider_number'] = rpt['provider_number'].astype(str).str.strip().str.zfill(6)
    rpt = rpt[['rpt_rec_num', 'provider_number', 'fy_bgn_dt', 'fy_end_dt', 'fi_creat_dt']].dropna(subset=['rpt_rec_num'])

    nmrc = pd.read_csv(nmrc_path, header=None, names=NMRC_COLS, dtype=str)
    nmrc['rpt_rec_num'] = pd.to_numeric(nmrc['rpt_rec_num'], errors='coerce')
    nmrc['itm_val_num'] = pd.to_numeric(nmrc['itm_val_num'], errors='coerce')

    base = rpt.copy()
    for var_name, (ws, line, col) in S10_VARS.items():
        sub = nmrc[
            (nmrc['wksht_cd'] == ws) &
            (nmrc['line_num'] == line) &
            (nmrc['clmn_num'] == col)
        ][['rpt_rec_num', 'itm_val_num']].rename(columns={'itm_val_num': var_name})
        base = base.merge(sub, on='rpt_rec_num', how='left')

    base['year'] = year
    return base

print('extract_s10_year defined.')

extract_s10_year defined.


In [23]:
uc_years = []

for yr in range(2011, 2019):
    print(f'Extracting S-10 for {yr}...', end=' ')
    df_yr = extract_s10_year(yr)
    if df_yr is not None:
        uc_years.append(df_yr)
        n_charges = df_yr['tot_uncomp_care_charges'].notna().sum()
        n_bd      = df_yr['bad_debt'].notna().sum()
        print(f'{len(df_yr):,} reports | charges={n_charges:,} | bad_debt={n_bd:,}')

uc_raw = pd.concat(uc_years, ignore_index=True)
print(f'\nTotal rows across all years: {len(uc_raw):,}')

Extracting S-10 for 2011... 6,143 reports | charges=4,310 | bad_debt=5,308
Extracting S-10 for 2012... 6,202 reports | charges=4,443 | bad_debt=5,172
Extracting S-10 for 2013... 6,144 reports | charges=4,435 | bad_debt=5,103
Extracting S-10 for 2014... 6,190 reports | charges=4,458 | bad_debt=5,129
Extracting S-10 for 2015... 6,199 reports | charges=4,400 | bad_debt=5,131
Extracting S-10 for 2016... 6,204 reports | charges=4,410 | bad_debt=4,839
Extracting S-10 for 2017... 6,121 reports | charges=4,449 | bad_debt=4,650
Extracting S-10 for 2018... 6,159 reports | charges=4,413 | bad_debt=4,624

Total rows across all years: 49,362


In [24]:
uc_raw['uncomp_care'] = (
    uc_raw['tot_uncomp_care_charges']
    - uc_raw['tot_uncomp_care_partial_pmts'].fillna(0)
    + uc_raw['bad_debt']
)

print(f'Rows with uncomp_care: {uc_raw["uncomp_care"].notna().sum():,}')
print(f'Mean uncomp_care 2014 (all reports): ${uc_raw[uc_raw["year"]==2014]["uncomp_care"].mean()/1e6:.2f}mm')

Rows with uncomp_care: 35,082
Mean uncomp_care 2014 (all reports): $28.23mm


In [25]:
uc_dedup1 = (
    uc_raw
    .sort_values(['provider_number', 'fy_bgn_dt', 'fy_end_dt', 'fi_creat_dt'],
                  ascending=[True, True, True, False])
    .drop_duplicates(subset=['provider_number', 'fy_bgn_dt', 'fy_end_dt'], keep='first')
    .copy()
)

print(f'After step-1 dedup: {len(uc_dedup1):,} rows')

uc_dedup1['time_diff'] = (uc_dedup1['fy_end_dt'] - uc_dedup1['fy_bgn_dt']).dt.days
uc_dedup1['n_rpts']    = uc_dedup1.groupby(['provider_number', 'year'])['provider_number'].transform('count')

unique1 = uc_dedup1[uc_dedup1['n_rpts'] == 1].copy()
dupes   = uc_dedup1[uc_dedup1['n_rpts'] > 1].copy()
dupes['total_days'] = dupes.groupby(['provider_number', 'year'])['time_diff'].transform('sum')

print(f'Unique reports:    {len(unique1):,}')
print(f'Duplicate reports: {len(dupes):,}')

After step-1 dedup: 49,362 rows
Unique reports:    47,715
Duplicate reports: 1,647


In [26]:
def na_sum(x):
    return np.nan if x.isna().all() else x.sum(skipna=True)

short = dupes[dupes['total_days'] < 370].copy()

if len(short) > 0:
    unique2 = (
        short
        .groupby(['provider_number', 'year'])
        .agg(
            fy_bgn_dt=('fy_bgn_dt', 'min'),
            fy_end_dt=('fy_end_dt', 'max'),
            uncomp_care=('uncomp_care', na_sum),
        )
        .reset_index()
    )
else:
    unique2 = pd.DataFrame()

long_d = dupes[dupes['total_days'] >= 370].copy()
long_d['max_days'] = long_d.groupby(['provider_number', 'year'])['time_diff'].transform('max')
long_d['max_end']  = long_d.groupby(['provider_number', 'year'])['fy_end_dt'].transform('max')

primary_mask = (
    (long_d['time_diff'] == long_d['max_days']) &
    (long_d['time_diff'] > 360) &
    (long_d['fy_end_dt'] == long_d['max_end'])
)
unique3 = long_d[primary_mask].drop_duplicates(subset=['provider_number', 'year']).copy()

used = set(zip(unique3['provider_number'], unique3['year']))
remainder = long_d[
    ~long_d.apply(lambda r: (r['provider_number'], r['year']) in used, axis=1)
].copy()

if len(remainder) > 0:
    remainder['uncomp_care'] = remainder['uncomp_care'] * (remainder['time_diff'] / remainder['total_days'])
    unique4 = (
        remainder
        .groupby(['provider_number', 'year'])
        .agg(
            fy_bgn_dt=('fy_bgn_dt', 'min'),
            fy_end_dt=('fy_end_dt', 'max'),
            uncomp_care=('uncomp_care', na_sum),
        )
        .reset_index()
    )
else:
    unique4 = pd.DataFrame()

print(f'Short dupe groups:    {len(unique2):,}')
print(f'Primary long reports: {len(unique3):,}')
print(f'Weighted avg groups:  {len(unique4):,}')

Short dupe groups:    161
Primary long reports: 590
Weighted avg groups:  54


In [27]:
keep_cols = ['provider_number', 'year', 'uncomp_care']

frames = [unique1[keep_cols]]
if len(unique2) > 0: frames.append(unique2[keep_cols])
if len(unique3) > 0: frames.append(unique3[keep_cols])
if len(unique4) > 0: frames.append(unique4[keep_cols])

uc_dedup = pd.concat(frames, ignore_index=True)
uc_dedup = uc_dedup[uc_dedup['uncomp_care'].notna()].copy()
uc_dedup['year']            = uc_dedup['year'].astype(int)
uc_dedup['provider_number'] = uc_dedup['provider_number'].astype(str).str.zfill(6)

print(f'Final deduped rows: {len(uc_dedup):,}')
print('\nMean uncomp_care by year (sanity check):')
print(
    uc_dedup.groupby('year')['uncomp_care']
    .agg(mean='mean', n='count')
    .assign(mean_mm=lambda d: d['mean'] / 1e6)
    [['mean_mm', 'n']]
    .round(2)
)
print('\nProfessor targets: 2011=34.76  2012=37.32  2013=39.37  2014=36.57  2015=33.19')

Final deduped rows: 34,576

Mean uncomp_care by year (sanity check):
      mean_mm     n
year               
2011    29.78  4262
2012    32.25  4348
2013    32.60  4338
2014    28.56  4362
2015    28.17  4323
2016    38.05  4290
2017    36.71  4361
2018    39.17  4292

Professor targets: 2011=34.76  2012=37.32  2013=39.37  2014=36.57  2015=33.19


In [28]:
hcris['year']    = hcris['year'].astype(int)
uc_dedup['year'] = uc_dedup['year'].astype(int)

hcris_uc = hcris.merge(
    uc_dedup[['provider_number', 'year', 'uncomp_care']],
    on=['provider_number', 'year'],
    how='left'
)

hcris_uc = (
    hcris_uc
    .dropna(subset=['uncomp_care'])
    .query('2011 <= year <= 2018')
    .copy()
    .reset_index(drop=True)
)

hcris_uc['uncomp_care_m'] = hcris_uc['uncomp_care'] / 1_000_000

print(f'Rows after merge & filter: {len(hcris_uc):,}')
print(f'Years:                     {sorted(hcris_uc["year"].unique())}')
print(f'Unique providers:          {hcris_uc["provider_number"].nunique():,}')

Rows after merge & filter: 32,655
Years:                     [np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018)]
Unique providers:          4,558


In [29]:
kff_raw = pd.read_csv(KFF_IN, skiprows=2)

kff = kff_raw[['Location',
               'Status of Medicaid Expansion Decision',
               'Expansion Implementation Date']].copy()
kff.columns = ['state_name', 'expansion_status', 'date_adopted']

kff = kff[kff['state_name'].notna()]
kff = kff[~kff['state_name'].str.startswith(
    ('Notes', 'Source', 'Footnote'), na=True
)]
kff = kff[kff['state_name'] != 'United States']
kff = kff[kff['state_name'] != 'District of Columbia']
kff = kff[kff['state_name'].str.len() > 2].copy()

kff['date_adopted'] = pd.to_datetime(kff['date_adopted'], errors='coerce')
kff['expand_year']  = kff['date_adopted'].dt.year
kff['expanded']     = kff['expand_year'].notna().astype(int)

print(f'KFF rows (50 states): {len(kff)}')
print(kff['expand_year'].value_counts().sort_index())

KFF rows (50 states): 64
expand_year
2,014.00    26
2,015.00     3
2,016.00     2
2,019.00     2
2,020.00     3
2,021.00     2
2,023.00     2
Name: count, dtype: int64


In [30]:
xwalk = pd.DataFrame({
    'state': [
        'AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA',
        'HI','ID','IL','IN','IA','KS','KY','LA','ME','MD',
        'MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ',
        'NM','NY','NC','ND','OH','OK','OR','PA','RI','SC',
        'SD','TN','TX','UT','VT','VA','WA','WV','WI','WY'
    ],
    'state_name': [
        'Alabama','Alaska','Arizona','Arkansas','California','Colorado',
        'Connecticut','Delaware','Florida','Georgia','Hawaii','Idaho',
        'Illinois','Indiana','Iowa','Kansas','Kentucky','Louisiana',
        'Maine','Maryland','Massachusetts','Michigan','Minnesota',
        'Mississippi','Missouri','Montana','Nebraska','Nevada',
        'New Hampshire','New Jersey','New Mexico','New York',
        'North Carolina','North Dakota','Ohio','Oklahoma','Oregon',
        'Pennsylvania','Rhode Island','South Carolina','South Dakota',
        'Tennessee','Texas','Utah','Vermont','Virginia','Washington',
        'West Virginia','Wisconsin','Wyoming'
    ]
})

hcris_named = hcris_uc.merge(xwalk, on='state', how='left')

merged = hcris_named.merge(
    kff[['state_name', 'expand_year', 'expanded', 'date_adopted']],
    on='state_name',
    how='left'
)

print(f'Merged shape:  {merged.shape}')
print(f'Unique states: {merged["state"].nunique()}')

Merged shape:  (32655, 29)
Unique states: 50


In [35]:
df = merged.copy()
df['year'] = df['year'].astype(int)

df['expand_ever'] = df['expanded'].fillna(0).astype(int)

df['treat'] = (
    df['expand_ever'].eq(1) &
    df['year'].ge(df['expand_year'].fillna(9999).astype(int))
).astype(int)

df['post'] = (df['year'] >= 2014).astype(int)

df['time_to_treat'] = np.where(
    df['expand_ever'] == 1,
    df['year'] - df['expand_year'].fillna(0).astype(int),
    0
)

df['time_to_treat_bin'] = (
    df['time_to_treat'].clip(lower=-4, upper=5).astype(int)
)

print('Variables created.')
print(df[['expand_ever','treat','post','time_to_treat','time_to_treat_bin']].describe().round(2))

Variables created.
       expand_ever     treat      post  time_to_treat  time_to_treat_bin
count    32,655.00 32,655.00 32,655.00      32,655.00          32,655.00
mean          0.81      0.38      0.62          -0.97              -0.49
std           0.39      0.48      0.48           3.35               2.42
min           0.00      0.00      0.00         -12.00              -4.00
25%           1.00      0.00      0.00          -3.00              -3.00
50%           1.00      0.00      1.00           0.00               0.00
75%           1.00      1.00      1.00           1.00               1.00
max           1.00      1.00      1.00           4.00               4.00


In [36]:
sample_A = df.copy()

sample_B = df[
    df['expand_year'].isna() |
    df['expand_year'].eq(2014)
].copy().reset_index(drop=True)

sample_B_2x2 = sample_B[
    sample_B['year'].isin([2012, 2015])
].copy().reset_index(drop=True)

print(f'Sample A (full panel):    {len(sample_A):,} rows | {sample_A["provider_number"].nunique():,} hospitals | {sample_A["state"].nunique()} states')
print(f'Sample B (2014 + never):  {len(sample_B):,} rows | {sample_B["provider_number"].nunique():,} hospitals | {sample_B["state"].nunique()} states')
print(f'Sample B 2x2 (2012+2015): {len(sample_B_2x2):,} rows | {sample_B_2x2["provider_number"].nunique():,} hospitals')

Sample A (full panel):    32,655 rows | 4,558 hospitals | 50 states
Sample B (2014 + never):  23,344 rows | 3,266 hospitals | 36 states
Sample B 2x2 (2012+2015): 5,844 rows | 3,093 hospitals


In [37]:
keep_cols = [
    'provider_number', 'year', 'state', 'state_name',
    'uncomp_care', 'uncomp_care_m',
    'expand_year', 'expand_ever', 'expanded', 'date_adopted',
    'post', 'treat', 'time_to_treat', 'time_to_treat_bin'
]

sample_A[keep_cols].to_csv(OUT_DIR / 'hcris_kff_full.csv', index=False)
sample_B[keep_cols].to_csv(OUT_DIR / 'hcris_kff_2014.csv', index=False)

print('=== OUTPUT FILES ===')
for fname, desc in [
    ('hcris_kff_full.csv', 'Full panel'),
    ('hcris_kff_2014.csv', '2014 + never expanders'),
]:
    fpath = OUT_DIR / fname
    tmp   = pd.read_csv(fpath)
    print(f'  {fname:<30s}  {len(tmp):>7,} rows   {desc}')

print('\n=== MEAN uncomp_care_m BY YEAR (full panel) ===')
print(
    sample_A.groupby('year')['uncomp_care_m']
    .agg(mean='mean', n='count')
    .round(2)
)

=== OUTPUT FILES ===
  hcris_kff_full.csv               32,655 rows   Full panel
  hcris_kff_2014.csv               23,344 rows   2014 + never expanders

=== MEAN uncomp_care_m BY YEAR (full panel) ===
      mean     n
year            
2011 30.74  4044
2012 33.13  4124
2013 33.67  4096
2014 29.48  4103
2015 29.22  4080
2016 39.61  4043
2017 38.00  4116
2018 40.66  4049
